# Benchmarking ML Classifiers for Heart Disease Prediction
**Course:** PMIT_6121: Machine Learning  
**Dataset:** CDC Personal Key Indicators of Heart Disease (Kaggle)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

DARK_BLUE = '#1A3A5C'; MID_BLUE = '#2E75B6'
ACCENT = '#E74C3C';    GREEN   = '#27AE60'
GOLD   = '#F39C12';    LGRAY   = '#F4F6F9'

plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,
    'axes.spines.right':False,'axes.facecolor':LGRAY,
    'figure.facecolor':'white','axes.grid':True,
    'grid.alpha':0.4,'grid.color':'white'})
print('Libraries loaded.')

## 2. Load Dataset

In [ ]:
# Download from Kaggle first:
# kaggle datasets download -d kamilpytlak/personal-key-indicators-of-heart-disease
df = pd.read_csv('heart_2020_cleaned.csv')
print(f'Shape: {df.shape}')
print(df['HeartDisease'].value_counts())
df.head(3)

## 3. EDA — Figure 1: Class Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))
fig.suptitle('Figure 1 — Target Class Distribution', fontsize=13, fontweight='bold', color=DARK_BLUE)

counts = df['HeartDisease'].value_counts().sort_index()
labels = ['No Heart Disease','Heart Disease']
colors = [MID_BLUE, ACCENT]

wedges,texts,autotexts = ax1.pie(counts, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':10})
for at in autotexts: at.set_color('white'); at.set_fontweight('bold')
ax1.set_title('Proportion', fontsize=11, color=DARK_BLUE)

bars = ax2.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar,val in zip(bars, counts.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2000,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold', color=DARK_BLUE)
ax2.set_ylabel('Count'); ax2.set_title('Record Counts', fontsize=11, color=DARK_BLUE)
plt.tight_layout(); plt.savefig('fig1_class_distribution.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. EDA — Figure 2: BMI Distribution by Class

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
fig.suptitle('Figure 2 — BMI Distribution by Heart Disease Status', fontsize=13, fontweight='bold', color=DARK_BLUE)

bmi_no  = df[df['HeartDisease']=='No']['BMI'].clip(10,60)
bmi_yes = df[df['HeartDisease']=='Yes']['BMI'].clip(10,60)

ax.hist(bmi_no,  bins=60, alpha=0.6, color=MID_BLUE, label='No Heart Disease', density=True)
ax.hist(bmi_yes, bins=60, alpha=0.6, color=ACCENT,   label='Heart Disease',    density=True)
ax.axvline(bmi_no.mean(),  color=MID_BLUE, linestyle='--', linewidth=1.8, label=f'Mean (No HD) = {bmi_no.mean():.1f}')
ax.axvline(bmi_yes.mean(), color=ACCENT,   linestyle='--', linewidth=1.8, label=f'Mean (HD)    = {bmi_yes.mean():.1f}')
ax.set_xlabel('BMI'); ax.set_ylabel('Density'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig2_bmi_distribution.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. EDA — Figure 3: Heart Disease Rate by Age Category

In [ ]:
age_cats = ['18-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65-69','70-74','75-79','80+']
df_tmp = df.copy(); df_tmp['HD_bin'] = (df_tmp['HeartDisease']=='Yes').astype(int)
age_hd = df_tmp.groupby('AgeCategory')['HD_bin'].mean()*100
age_hd = age_hd.reindex(age_cats)

fig, ax = plt.subplots(figsize=(11,4))
fig.suptitle('Figure 3 — Heart Disease Prevalence by Age Category', fontsize=13, fontweight='bold', color=DARK_BLUE)
bars = ax.bar(age_hd.index, age_hd.values,
              color=[ACCENT if v>10 else MID_BLUE for v in age_hd.values], edgecolor='white', linewidth=1.2)
for bar,val in zip(bars, age_hd.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{val:.1f}%',
            ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xlabel('Age Category'); ax.set_ylabel('Heart Disease Rate (%)')
ax.tick_params(axis='x', rotation=30)
ax.legend(handles=[mpatches.Patch(color=MID_BLUE,label='Rate ≤ 10%'),
                   mpatches.Patch(color=ACCENT,  label='Rate > 10%')], fontsize=9)
plt.tight_layout(); plt.savefig('fig3_age_prevalence.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. EDA — Figure 4: Risk Factors

In [ ]:
no  = df[df['HeartDisease']=='No']
yes = df[df['HeartDisease']=='Yes']
factors = ['Smoking','Diabetic','PhysicalActivity\n(Inactive)']
no_vals  = [(no['Smoking']=='Yes').mean()*100,  (no['Diabetic']=='Yes').mean()*100,  (no['PhysicalActivity']=='No').mean()*100]
yes_vals = [(yes['Smoking']=='Yes').mean()*100, (yes['Diabetic']=='Yes').mean()*100, (yes['PhysicalActivity']=='No').mean()*100]

fig, ax = plt.subplots(figsize=(9,4))
fig.suptitle('Figure 4 — Risk Factor Prevalence by Class', fontsize=13, fontweight='bold', color=DARK_BLUE)
x = np.arange(len(factors)); w = 0.35
b1 = ax.bar(x-w/2, no_vals,  w, label='No Heart Disease', color=MID_BLUE, edgecolor='white')
b2 = ax.bar(x+w/2, yes_vals, w, label='Heart Disease',    color=ACCENT,   edgecolor='white')
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(factors); ax.set_ylabel('Prevalence (%)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig4_risk_factors.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Preprocessing

In [ ]:
df2 = df.copy()
df2['HeartDisease'] = df2['HeartDisease'].map({'Yes':1,'No':0})
cat_cols = df2.select_dtypes(include='object').columns.tolist()
df2 = pd.get_dummies(df2, columns=cat_cols, drop_first=True)

X = df2.drop('HeartDisease', axis=1)
y = df2['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'Train: {X_train_s.shape}  |  Test: {X_test_s.shape}')

## 8. Model Training

In [ ]:
# SVM
svm_lin = SVC(kernel='linear', C=1.0, random_state=42).fit(X_train_s, y_train)
svm_rbf = SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42).fit(X_train_s, y_train)
y_svm_lin = svm_lin.predict(X_test_s)
y_svm_rbf = svm_rbf.predict(X_test_s)

# KNN
knn_preds = {}
for k in [3,5,11]:
    knn_preds[k] = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train).predict(X_test_s)
best_k = max(knn_preds, key=lambda k: f1_score(y_test, knn_preds[k]))
y_knn = knn_preds[best_k]

# Naive Bayes
y_gnb = GaussianNB().fit(X_train_s, y_train).predict(X_test_s)
print(f'Best K = {best_k}')

## 9. Figure 5: KNN K-value Comparison

In [ ]:
k_vals = [3,5,11]
acc_k = [accuracy_score(y_test, knn_preds[k]) for k in k_vals]
rec_k = [recall_score(y_test,   knn_preds[k]) for k in k_vals]
f1_k  = [f1_score(y_test,       knn_preds[k]) for k in k_vals]

fig, ax = plt.subplots(figsize=(7,4))
fig.suptitle('Figure 5 — KNN Performance Across K Values', fontsize=13, fontweight='bold', color=DARK_BLUE)
x = np.arange(len(k_vals)); w = 0.25
ax.bar(x-w, acc_k, w, label='Accuracy', color=MID_BLUE, edgecolor='white')
ax.bar(x,   rec_k, w, label='Recall',   color=ACCENT,   edgecolor='white')
ax.bar(x+w, f1_k,  w, label='F1-Score', color=GREEN,    edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels([f'K={k}' for k in k_vals])
ax.set_ylabel('Score'); ax.set_ylim(0,1.05); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig5_knn_k_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

## 10. Metrics Summary

In [ ]:
def get_metrics(yt, yp, label):
    return {'Model':label,'Accuracy':round(accuracy_score(yt,yp),4),
            'Precision':round(precision_score(yt,yp),4),
            'Recall':round(recall_score(yt,yp),4),'F1-Score':round(f1_score(yt,yp),4)}

summary = pd.DataFrame([
    get_metrics(y_test, y_svm_lin, 'SVM (Linear, C=1.0)'),
    get_metrics(y_test, y_svm_rbf, 'SVM (RBF,    C=1.0)'),
    get_metrics(y_test, y_knn,     f'KNN (K={best_k})'),
    get_metrics(y_test, y_gnb,     'Naive Bayes (Gaussian)'),
])
print(summary.to_string(index=False))
summary

## 11. Figure 6: Cross-Model Comparison

In [ ]:
model_names = ['SVM\nLinear','SVM\nRBF',f'KNN\nK={best_k}','Naive\nBayes']
all_preds   = [y_svm_lin, y_svm_rbf, y_knn, y_gnb]
metrics_dict = {
    'Accuracy':  [accuracy_score(y_test,p)  for p in all_preds],
    'Precision': [precision_score(y_test,p) for p in all_preds],
    'Recall':    [recall_score(y_test,p)    for p in all_preds],
    'F1-Score':  [f1_score(y_test,p)        for p in all_preds],
}
cols = [MID_BLUE, MID_BLUE, GREEN, GOLD]
fig, axes = plt.subplots(1,4, figsize=(14,4))
fig.suptitle('Figure 6 — Comparative Model Performance', fontsize=13, fontweight='bold', color=DARK_BLUE)
for ax,(metric,vals) in zip(axes, metrics_dict.items()):
    bars = ax.bar(model_names, vals, color=cols, edgecolor='white', linewidth=1.2)
    bars[int(np.argmax(vals))].set_edgecolor(ACCENT); bars[int(np.argmax(vals))].set_linewidth(3)
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_title(metric, fontsize=11, fontweight='bold', color=DARK_BLUE)
    ax.set_ylim(0, min(1.15, max(vals)*1.25))
plt.tight_layout(); plt.savefig('fig6_model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

## 12. Figure 7: Confusion Matrices

In [ ]:
models = [('SVM (Linear)',y_svm_lin),('SVM (RBF)',y_svm_rbf),(f'KNN (K={best_k})',y_knn),('Naive Bayes',y_gnb)]
fig, axes = plt.subplots(1,4, figsize=(20,4))
fig.suptitle('Figure 7 — Confusion Matrices for All Classifiers', fontsize=13, fontweight='bold', color=DARK_BLUE, y=1.02)
for ax,(title,yp) in zip(axes, models):
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['No HD','HD'], yticklabels=['No HD','HD'],
                linewidths=0.5, linecolor='white', annot_kws={'size':11,'weight':'bold'})
    ax.set_title(title, fontsize=11, fontweight='bold', color=DARK_BLUE, pad=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig('fig7_confusion_matrices.png', dpi=150, bbox_inches='tight'); plt.show()